# ATML PA3 – GPU runner (Colab / Kaggle)

Thin driver only: every experiment is a module in the repo. `outputs/` (checkpoints, merged models, caches) and
`results/` are symlinked to persistent storage, and every script resumes / skips finished work when re-run.

Rough budget: Task 1 needs five 1,250-update runs (Full FT on a T4 is the slowest; an L4/A100 is strongly
recommended) plus seven evaluation passes; Task 2 one 4,500-update run; Task 3 is CPU except the 50-question live
experiment.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
REPO_URL = "https://github.com/ahmedmahni07-commits/27100150_PA3.git"

import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if ON_KAGGLE else "/content"
PERSIST = "/kaggle/working/persist" if ON_KAGGLE else "/content/drive/MyDrive/ATML_PA3"
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
for d in ["outputs", "results", "data_large"]:
    os.makedirs(f"{PERSIST}/{d}", exist_ok=True)
print("Kaggle" if ON_KAGGLE else "Colab", "->", PERSIST)

In [ ]:
%cd $WORK
!rm -rf pa3 && git clone -q $REPO_URL pa3
%cd $WORK/pa3
!rm -rf outputs && ln -s $PERSIST/outputs outputs
!mkdir -p $PERSIST/results && cp -rn results/. $PERSIST/results/ ; rm -rf results && ln -s $PERSIST/results results
# large data folders live on Drive too
!for d in task2_raw task2_images ATML-PA3-Data; do mkdir -p $PERSIST/data_large/$d; ln -sfn $PERSIST/data_large/$d data/$d; done
!pip -q install -r requirements.txt
!python -m tests.test_task1 && python -m tests.test_task3

## Task 1 – fine-tuning

In [ ]:
!python -m task1_ft.data --config configs/task1.yaml
!python -m task1_ft.diagnostics --config configs/task1.yaml probes
!python -m task1_ft.evaluate --config configs/task1.yaml --condition baseline

In [ ]:
# one cell per run so a disconnect only costs the current run (re-running resumes)
!python -m task1_ft.train --config configs/task1.yaml --method full_ft

In [ ]:
!python -m task1_ft.train --config configs/task1.yaml --method l2sp

In [ ]:
!python -m task1_ft.train --config configs/task1.yaml --method lora4
!python -m task1_ft.train --config configs/task1.yaml --method lora16
!python -m task1_ft.train --config configs/task1.yaml --method oplora16

In [ ]:
!python -m task1_ft.evaluate --config configs/task1.yaml --condition full_ft --update 625
for m in ["full_ft", "l2sp", "lora4", "lora16", "oplora16"]:
    !python -m task1_ft.evaluate --config configs/task1.yaml --condition {m}

In [ ]:
!python -m task1_ft.diagnostics --config configs/task1.yaml cache
!python -m task1_ft.diagnostics --config configs/task1.yaml analyze
!python -m task1_ft.diagnostics --config configs/task1.yaml verify_oplora
!python -m task1_ft.qualitative --config configs/task1.yaml
!python -m task1_ft.qualitative --config configs/task1.yaml --annotation-sheet
!python -m task1_ft.summarize --config configs/task1.yaml

## Task 2 – VLM

In [ ]:
!python -m task2_vlm.data download
!python -m task2_vlm.data select
!python -m task2_vlm.data images
!python -m task2_vlm.features --config configs/task2.yaml

In [ ]:
!python -m task2_vlm.train --config configs/task2.yaml

In [ ]:
!python -m task2_vlm.evaluate --config configs/task2.yaml
!python -m task2_vlm.summarize --config configs/task2.yaml

In [ ]:
# optional 49-token run
# !python -m task2_vlm.train --config configs/task2.yaml --n-vis 49
# !python -m task2_vlm.evaluate --config configs/task2.yaml --n-vis 49

## Task 3 – test-time compute (copy ATML-PA3-Data into data/ATML-PA3-Data first)

In [ ]:
!wget -q -O data/mathvista_testmini.json https://raw.githubusercontent.com/lupantech/MathVista/main/data/testmini.json
!python -m task3_ttc.data inspect
!python -m task3_ttc.data build && python -m task3_ttc.grading
!python -m task3_ttc.offline && python -m task3_ttc.signals && python -m task3_ttc.adaptive

In [ ]:
!python -m task3_ttc.live generate
!python -m task3_ttc.live summarize
!python -m task3_ttc.plots